# Создание и заполнение данных БД Postgre

In [2]:
%pip install python-dotenv psycopg2-binary
%pip install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import psycopg2
import pandas as pd
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [4]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [6]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [17]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [18]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


In [10]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура, специалитет, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [104]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


In [105]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost",
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

conn.autocommit = True

Успешное подключение к базе данных!


In [106]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
cursor.execute(query)
rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]
print(f"Колонки: {columns}\n")

for row in rows:
    print(row)

Колонки: ['courseid', 'userid', 'num_week', 's_all', 's_all_avg', 's_course_viewed', 's_course_viewed_avg', 's_q_attempt_viewed', 's_q_attempt_viewed_avg', 's_a_course_module_viewed', 's_a_course_module_viewed_avg', 's_a_submission_status_viewed', 's_a_submission_status_viewed_avg', 'namer_level', 'name_vatt', 'depart', 'name_osno', 'name_formopril', 'leveled', 'num_sem', 'kurs', 'date_vatt']

(84335, 29750, 9, 0, '3', 0, '2,75', 0, '0', 0, '0', 0, '0', '5', 'Экзамен', 14, '1', '1', '1', 4, 3, '20.06.2022')
(72359, 31929, 20, 0, '3.9333', 0, '1,3333', 0, '0', 0, '1,1333', 0, '0.8667', '4', 'Экзамен', 31, '2', '2', '1', 4, 3, '29.06.2022')
(84989, 27532, 14, 0, '1.4444', 0, '0', 0, '0', 0, '0,3333', 0, '0.3333', '5', 'Экзамен', 23, '2', '2', '1', 6, 4, '28.06.2022')
(79466, 25887, 8, 19, '43.6667', 6, '13,6667', 0, '0', 6, '10,6667', 6, '8.6667', '3', 'Экзамен', 24, '2', '1', '2', 6, 4, '24.06.2022')
(72457, 36370, 18, 0, '0.8462', 0, '0,6154', 0, '0', 0, '0', 0, '0', '4', 'Экзамен', 19

In [21]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_18856\1171977693.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,depart,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt
0,87077,28936,18,14,"6,3077",2,"1,1538",0,0,5,...,"1,5385",4,Экзамен,23,1,1,1,4,3,10.06.2022
1,75714,7450,27,0,"1,4091",0,"0,3182",0,0,0,...,"0,1818",5,Экзамен,7,1,2,2,12,7,20.06.2022
2,72126,36406,17,0,"0,0833",0,0,0,0,0,...,0,3,Экзамен,3,1,2,1,2,2,23.06.2022
3,84825,25087,9,0,2,0,"0,5",0,0,0,...,"0,75",4,Экзамен,20,1,1,2,6,4,27.06.2022
4,78285,23279,15,0,"0,3",0,"0,1",0,0,0,...,"0,1",4,Экзамен,11,2,2,1,8,5,22.06.2022
5,72359,32360,9,0,0,0,0,0,0,0,...,0,4,Экзамен,19,2,2,1,4,3,29.06.2022
6,84879,23562,20,0,0,0,0,0,0,0,...,0,5,Экзамен,15,2,2,2,8,5,21.06.2022
7,84879,18775,9,0,0,0,0,0,0,0,...,0,4,Экзамен,15,1,2,2,10,6,20.06.2022
8,78572,29819,20,22,"23,4667",8,"5,8667",0,"5,0667",6,...,"2,7333",5,Экзамен,16,1,1,1,4,3,22.06.2022
9,79114,24979,10,28,"19,2",4,"3,6",0,0,9,...,"4,4",3,Экзамен,14,2,1,1,6,4,04.07.2022


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [22]:
query = """
    SELECT 
        COUNT(DISTINCT depart) AS departments_count
    FROM USER_LOGS
    WHERE courseid IS NOT NULL
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_18856\3608507271.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,departments_count
0,43


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводить сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [25]:
query = """
    SELECT 
        DEPARTMENTS.name AS DEPTNAME,
        COUNT(USER_LOGS.courseid) AS COUNTCOURSE
    FROM USER_LOGS INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    GROUP BY DEPARTMENTS.name
    ORDER BY COUNTCOURSE DESC
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_18856\1262581724.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,deptname,countcourse
0,МиХТ,25296
1,ДиСО,25176
2,ЛиП,19008
3,РМПИ,17856
4,ГМДиОПИ,16968
5,ТОМ,16704
6,БИиИТ,16176
7,ПОиД,14760
8,АЭПиМ,14232
9,ЛиУТС,13440


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [32]:
query = """

    SELECT 
        COUNT(DEPTCOUNT.courseid)
    FROM 
    (
        SELECT
            USER_LOGS.courseid,
            COUNT(DISTINCT USER_LOGS.depart) AS departments_count
        FROM USER_LOGS
        WHERE courseid IS NOT NULL
        GROUP BY USER_LOGS.courseid
        HAVING COUNT(DISTINCT depart) > 1
    ) AS DEPTCOUNT 
"""



df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\866560846.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count
0,60


In [42]:
#получили курсы, на которые закреплены несколько кафедр
query = """
    SELECT
        USER_LOGS.courseid AS COURSEID,
        COUNT(DISTINCT USER_LOGS.depart) AS DEPCOUNT
    FROM
        USER_LOGS
    GROUP BY USER_LOGS.courseid
    HAVING COUNT(DISTINCT USER_LOGS.depart) > 1
"""

#мы должны теперь найти уникальные кафедры, сравнивая их курсы с курсами, имеющими несколько кафедр
#тем самым на каждый курс получим кафедры, которые закреплены именно на один курс.
query = """

    WITH COURSES AS 
    (
	SELECT
		USER_LOGS.courseid AS COURSEID,
		COUNT(DISTINCT USER_LOGS.depart) AS DEPCOUNT
	FROM
		USER_LOGS
	GROUP BY USER_LOGS.courseid
	HAVING COUNT(DISTINCT USER_LOGS.depart) > 1
    )

    SELECT DISTINCT
	    DEPARTMENTS.name,
	    UL.courseid
    FROM USER_LOGS AS UL
	    INNER JOIN DEPARTMENTS ON UL.DEPART = DEPARTMENTS.ID
	    INNER JOIN COURSES ON UL.COURSEID = COURSES.COURSEID
    ORDER BY UL.courseid
"""


df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_18856\461634636.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,name,courseid
0,ГМиТТК,71495
1,ПиСЗ,71495
2,УиИС,71495
3,ПиСЗ,71508
4,УиИС,71508
...,...,...
165,ПМиИ,88653
166,БИиИТ,88659
167,ПМиИ,88659
168,ЛиП,88888


## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [51]:
conn.rollback()

In [ ]:
query = """
    SELECT 
        namer_level AS namer_level,
        COUNT(userid) AS count
    FROM user_logs
    GROUP BY namer_level
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\4022336865.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,namer_level,count
0,2,44184


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за весь период обучения).

In [28]:
conn.rollback()

In [31]:
# query = """

#     WITH MAX_COUNT_LOGS AS
#     (
#         SELECT
#             userid AS max_user,
#             MAX(USER_LOGS.s_all) AS max_logs
#         FROM USER_LOGS
#         GROUP BY max_user
       
#     )


#     SELECT 
#         userid AS user,
#         s_all AS count_logs
#     FROM
#         USER_LOGS AS UL
#             INNER JOIN MAX_COUNT_LOGS AS MCL ON UL.userid = MCL.max_user
# """

#в целом можно и limit использовать, но можно и cte.

query = """
   SELECT 
       userid AS user,
       s_all AS count_logs
   FROM
       USER_LOGS
   ORDER BY s_all DESC
   LIMIT 1
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\2647836617.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,user,count_logs
0,21606,3940


## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [ ]:
conn.rollback()

In [32]:
query = """
    SELECT
        num_week,
        AVG(s_all)
    FROM USER_LOGS
    GROUP BY num_week
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\3882656882.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,num_week,avg
0,6,13.796607
1,7,9.616142
2,8,8.028543
3,9,9.393296
4,10,8.208546
5,11,10.022001
6,12,9.381716
7,13,10.014011
8,14,9.860178
9,15,10.353694


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [78]:
conn.rollback()

In [80]:
query = """
    SELECT
        COUNT(DISTINCT USER_LOGS.userid) AS count_students,
        DEPARTMENTS.name
    FROM USER_LOGS
        INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    WHERE namer_level = '5'
    GROUP BY DEPARTMENTS.name
    ORDER BY count_students DESC
    LIMIT 1
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\2177089847.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count_students,name
0,310,ДиСО


In [79]:
#Список кафедр и количество студентов в каждой из них
query = """

    SELECT
        COUNT(DISTINCT userid),
        depart
    FROM USER_LOGS
    WHERE namer_level = '2'
    GROUP BY depart
"""

query = """
    SELECT
        COUNT(DISTINCT USER_LOGS.userid) AS count_students,
        DEPARTMENTS.name
    FROM USER_LOGS
        INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    WHERE namer_level = '2'
    GROUP BY DEPARTMENTS.name
    ORDER BY count_students DESC
    LIMIT 1
"""



df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\270051105.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count_students,name
0,72,Эконом.


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [91]:
conn.rollback()

In [108]:
conn.rollback()

query = """
    SELECT
        name_osno,
        week_number,
        avg_s_all,
        avg_s_course_viewed,--это по сути активность студента, у которого
         -- name_osno бюджет или контракт / вся активность студентов
        avg_s_q_attempt_viewed
    FROM USER_LOGS
    WHERE namer_level = '2'
    GROUP BY depart
"""

#получили самую насыщенную по событиям неделю
query = """
    SELECT
        name_osno,
        num_week AS week_number,
        s_all_avg AS week_events
    FROM USER_LOGS
    ORDER BY s_all_avg DESC
    LIMIT 1
"""

query = """
    SELECT
        name_osno,
        SUM(s_all_avg::REAL)
    FROM USER_LOGS
    WHERE num_week = 27
    GROUP BY name_osno
    
"""




df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_3064\1621615298.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,name_osno,sum
0,1,157956.25
1,2,53268.47
